In the following, we report a template for the final project. Feel free to modify it to better fit your own project.

# **Project Title**
**Per-Head Heavy-Hitter Oracle (PH-H2O)**

## **Abstract**
KV cache compression has been widely applied across domains such as natural language processing, computer vision, and other tasks. In this project, I reproduced four baselines — full cache (no eviction), LRU KV cache, Heavy-Hitter Oracle (H2O per-layer), and random eviction cache — and compared them against an H2O variant using a per-head KV cache method. Evaluation was conducted on an NVIDIA H100 GPU with 80 GB of memory on the CCDB Trillium cluster. The original H2O paper claims that retaining only 20% of the KV cache achieves approximately 80% accuracy or coverage on the CNN/DailyMail dataset. Our evaluation reproduces this result. Furthermore, the proposed per-head KV cache method achieves a 6% improvement over the per-layer H2O baseline at a 20% budget.



## **Introduction**

### **Problem Statement and Motivations**

Existing KV cache eviction policies, such as the standard H2O (Heavy-Hitter Oracle), typically compute eviction scores by aggregating importance metrics across all attention heads within a layer. While computationally efficient, this per-layer agreement creates a "**consensus bottleneck.**"

Because different heads attend to different semantic or syntactic features, a specific token (e.g., index 9) may be a "Heavy-Hitter" for Head 1 but irrelevant for Heads 2 and 3. In a shared eviction scheme, the low scores from Heads 2 and 3 can lead to the eviction of index 9, causing a localized loss of critical information for Head 1 and ultimately degrading model accuracy.

<figure style="text-align: center">
  <img src="../resource/attention.png" width="900">
  <figcaption>Figure 1: Attention mechanism</figcaption>
</figure>

Here is an example in the figure 1. The sequence "The cat sat on the mat near the TV". Assuming 4 heads are focusing on suj. and verb. , head 2 on article and det., head 3 on location and head 4 focusing on preposition. The per layer score will keep "The" token in Top 1 position but it is not top1 for all heads just an average result. Therefore, by using per head cache, each head can just reserve the corresponding top k value instead of using per-layer consensus.

### **Proposed solution**
To address this, I propose an incremental per-head eviction strategy ---- **Per-Head Heavy-Hitter Oracle (PH-H2O)**. Instead of a shared eviction index, each attention head maintains its own independent KV cache metadata and eviction policy.

This method has following advantages :
By decoupling the eviction logic to ensure that "specialist" heads can retain

1.   **Finer granularity**.the tokens most relevant to their specific functional role without being overridden by the "average" importance of the layer.
2.   **Same budget cache size but accuracy is improved**. Even when the total cache capacity (memory footprint) remains identical to the baseline, the information density of the cache increases because each head's budget is spent only on its most relevant tokens.
3.   **Minimal overhead.** Since H2O scores are already calculated per-head during the attention pass, the primary cost is maintaining head-specific bitmasks or index lists, which is negligible compared to the accuracy recovery.



<div style="display:flex; justify-content:center;">
<table style="border-collapse:collapse; font-family:'Segoe UI',Arial,sans-serif; font-size:14px;">
  <thead>
    <tr style="background:#1a73e8; color:#fff;">
      <th style="padding:10px 14px; text-align:left; font-weight:600; white-space:nowrap;">Policy</th>
      <th style="padding:10px 14px; text-align:left; font-weight:600; white-space:nowrap;">Granularity</th>
      <th style="padding:10px 14px; text-align:left; font-weight:600; white-space:nowrap;">Eviction Logic</th>
      <th style="padding:10px 14px; text-align:left; font-weight:600; white-space:nowrap;">Memory Footprint</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background:#fff;">
      <td style="padding:9px 14px; border-bottom:1px solid #e8eaed; white-space:nowrap;"><code>Full Cache</code></td>
      <td style="padding:9px 14px; border-bottom:1px solid #e8eaed; text-align:center;">—</td>
      <td style="padding:9px 14px; border-bottom:1px solid #e8eaed; white-space:nowrap;">No eviction</td>
      <td style="padding:9px 14px; border-bottom:1px solid #e8eaed; white-space:nowrap;">Full sequence</td>
    </tr>
    <tr style="background:#f8f9fa;">
      <td style="padding:9px 14px; border-bottom:1px solid #e8eaed; white-space:nowrap;"><code>Random</code></td>
      <td style="padding:9px 14px; border-bottom:1px solid #e8eaed; text-align:center;">Layer</td>
      <td style="padding:9px 14px; border-bottom:1px solid #e8eaed; white-space:nowrap;">Random selection</td>
      <td style="padding:9px 14px; border-bottom:1px solid #e8eaed; white-space:nowrap;">Fixed budget</td>
    </tr>
    <tr style="background:#fff;">
      <td style="padding:9px 14px; border-bottom:1px solid #e8eaed; white-space:nowrap;"><code>Local (LRU)</code></td>
      <td style="padding:9px 14px; border-bottom:1px solid #e8eaed; text-align:center;">Layer</td>
      <td style="padding:9px 14px; border-bottom:1px solid #e8eaed; white-space:nowrap;">Keep recent tokens</td>
      <td style="padding:9px 14px; border-bottom:1px solid #e8eaed; white-space:nowrap;">Fixed budget</td>
    </tr>
    <tr style="background:#f8f9fa;">
      <td style="padding:9px 14px; border-bottom:1px solid #e8eaed; white-space:nowrap;"><code>H2O (Standard)</code></td>
      <td style="padding:9px 14px; border-bottom:1px solid #e8eaed; text-align:center;">Layer</td>
      <td style="padding:9px 14px; border-bottom:1px solid #e8eaed; white-space:nowrap;">Recent + Heavy Hitter</td>
      <td style="padding:9px 14px; border-bottom:1px solid #e8eaed; white-space:nowrap;">Fixed budget</td>
    </tr>
    <tr style="background:#fff;">
      <td style="padding:9px 14px; white-space:nowrap;"><code>H2O (Per-head)</code></td>
      <td style="padding:9px 14px; text-align:center;">Head</td>
      <td style="padding:9px 14px; white-space:nowrap;">Recent + Heavy Hitter</td>
      <td style="padding:9px 14px; white-space:nowrap;">Fixed budget</td>
    </tr>
  </tbody>
</table>
</div>


**Same total budget — different selection granularity**
1. Per-layer: one global ranking loses head-specific patterns
2. Per-head: each head keeps what it actually attends to




## **Methodology**

<p>This project investigates KV cache compression for large language models, evaluating the Heavy Hitter Oracle (H2O) eviction strategy on <b>Llama-3.2-1B-Instruct</b> for abstractive summarization.</p>

<h3>Model and dataset</h3>
<p>The algorithm is evaluated on the <b>CNN/DailyMail 3.0.0 benchmark</b> using Llama-3.2-1B-Instruct — a 16-layer decoder-only model with Grouped Query Attention (GQA): 32 query heads and 8 KV heads. All experiments use a maximum input length of 4096 tokens and a fixed KV cache budget of 512 tokens.</p>

<h3>KV cache eviction policies</h3>
<p>four eviction strategies implemented and compared :</p>
<ul>
  <li><b>Full cache:</b> retains all KV pairs with no eviction. Upper-bound baseline.</li>
  <li><b>Random:</b> randomly selects tokens within a fixed budget. Lower-bound baseline.</li>
  <li><b>Local / sliding window:</b> retains only the most recent tokens (<code>budget = recent</code>). Pure recency-based cache with no heavy hitter tracking.</li>
  <li><b>H2O:</b> combines a recent window with attention-guided heavy hitters. Budget satisfies <code>total = HH + recent</code>.</li>
</ul>

<h3>H2O eviction: per-layer vs per-head</h3>
<p>By introducing a key design variant — whether heavy hitter selection operates at the layer level or the head level.</p>
<ul>
  <li><b>Per-layer:</b> attention scores are averaged across all KV heads, and a single shared top-k selection is applied uniformly to all heads. It's imple and efficient, but loses head-specific attention patterns.</li>
  <li><b>Per-head:</b> each KV head independently computes its own top-k heavy hitters, preserving head-specific linguistic patterns. The total budget is divided equally among heads: <code>per_head_budget = total_budget / num_kv_heads</code>.</li>
</ul>

<div style="display: flex; gap: 20px; justify-content: center; margin: 1rem 0;">
  <figure style="text-align: center; flex: 1;">
    <img src="../resource/Per%20layer.png" width="700">
    <figcaption>Figure 2: Per-layer KV cache demonstration</figcaption>
  </figure>
  <figure style="text-align: center; flex: 1;">
    <img src="../resource/per%20head.png" width="700">
    <figcaption>Figure 3: Per-head KV cache demonstration</figcaption>
  </figure>
</div>

<h3>GQA handling</h3>
<p>Because Llama-3.2-1B uses GQA with a 4:1 query-to-KV ratio, attention weights are reshaped and averaged over the 4 query heads sharing each KV head before computing accumulated scores:</p>
<pre><code>score = raw_score.view(NUM_KV_HEADS, GQA_GROUP, -1).mean(dim=1)</code></pre>

<h3>Prefill eviction</h3>
<p>Eviction is applied via a forward hook on <code>model.model</code>, triggering after each full forward pass. The hook inspects the <code>DynamicCache</code> and prunes KV entries whenever the sequence length exceeds the budget, keeping the top heavy hitters plus the most recent tokens.</p>





<h3>Implementation</h3>

Here is the flow chart of the implementation:

<div style="display: flex; gap: 24px; justify-content: center; align-items: flex-start;">
  <figure style="text-align: center; flex: 1; margin: 0;">
    <img src="../resource/flow chart.png" width="700">
    <figcaption>Figure 3: Per-head KV cache demonstration</figcaption>
  </figure>

  <figure style="text-align: center; flex: 1; margin: 0;">
    <img src="../resource/llama model.png" width="700">
    <figcaption>Figure 4: Llama 3.2-1B-Instruct and Monkey patch</figcaption>
  </figure>
</div>


In this section, you can add **text** and **figures**. For instance, it is strongly suggested to add a picture of the best machine learning model that you implemented to solve your problem (and describe it).


## **Experimental Setup**
Describe the datasets used for your experiments. List the machine learning techniques used to solve your problem and report the corresponding hyperparameters.

<h3>Hardware and environment</h3>
<p>All experiments were conducted on a single <b>NVIDIA H100 GPU with 80 GB of memory</b> on CCDB trillium cluster. The model was loaded in <code>bfloat16</code> precision using HuggingFace Transformers with <code>attn_implementation="eager"</code> to enable attention weight extraction.</p>

<h3>Model</h3>
<p>I use <b>Llama-3.2-1B-Instruct</b> (<code>meta-llama/Llama-3.2-1B-Instruct</code>) with the following configuration:</p>
<ul>
  <li>Hidden size: 2048, 16 transformer layers</li>
  <li>32 query heads, 8 KV heads (GQA with group size 4)</li>
</ul>

<h3>Dataset</h3>
<p>I evaluate on the following dataset</p>
<table border="1" cellpadding="8" cellspacing="0" style="border-collapse: collapse; width: 100%;">
  <thead>
    <tr style="background-color: #f2f2f2;">
      <th>Dataset</th>
      <th>Source</th>
      <th>Task</th>
      <th>Input Field</th>
      <th>Reference Field</th>
      <th>max_seq_len</th>
      <th>cache_size</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><b>CNN/DailyMail</b> 3.0.0</td>
      <td>HuggingFace <code>cnn_dailymail</code></td>
      <td>Summarization</td>
      <td><code>article</code></td>
      <td><code>highlights</code></td>
      <td>3000</td>
      <td>512</td>
    </tr>
    <tr>
      <td><b>GovReport</b></td>
      <td>LongBench <code>gov_report</code></td>
      <td>Summarization</td>
      <td><code>context</code></td>
      <td><code>answers</code></td>
      <td>3000</td>
      <td>512</td>
    </tr>
    <tr>
      <td><b>QMSum</b></td>
      <td>LongBench <code>qmsum</code></td>
      <td>Summarization</td>
      <td><code>context</code></td>
      <td><code>answers</code></td>
      <td>3000</td>
      <td>512</td>
    </tr>
    <tr>
      <td><b>VCSum</b></td>
      <td>LongBench <code>vcsum</code></td>
      <td>Summarization</td>
      <td><code>context</code></td>
      <td><code>answers</code></td>
      <td>3000</td>
      <td>512</td>
    </tr>
  </tbody>
</table>
<p>Here is an example of <b>CNN/DailyMail 3.0.0</b> validation split. Each article is truncated to a maximum of 4096 tokens:</p>

Here is what does each record in the CNN dataset:


<code>
Article: "LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20......".


Highlights: "Harry Potter star Daniel Radcliffe gets £20M fortune as he turns 18 Monday . Young actor.......".
</code>

The prompt used is :
<code>Prompt:"Summarize this article in 2-3 sentences:".</code>

<p>A maximum of 100 new tokens can be generated.</p>

<h3>KV cache budget</h3>
<p>The total KV cache size is fixed at <b>512 tokens</b>. Sweep the following hyperparameters:</p>
<ul>
  <li><code>budget_ratio</code>: 0.1 to 0.8 (fraction of 512 tokens to keep)</li>
  <li><code>recent_ratio</code>: 0.1 to 0.5 (fraction reserved for the recency window)</li>
  <li>H2O strategy: <code>per_head</code> vs <code>layer_shared</code></li>
</ul>

<h3>Baselines</h3>
<ul>
  <li><b>Full cache:</b> no eviction, retains all tokens</li>
  <li><b>Random eviction:</b> uniformly random token selection at each budget level</li>
  <li><b>Local (sliding window):</b> retains only the most recent tokens (<code>budget_ratio = recent_ratio</code>)</li>
  <li><b>H2O (per layer):</b></li>
  <li><b>PH H2O (per head H2O):</b></li>
</ul>

<h3>Evaluation metric</h3>
<p>All strategies are evaluated using <b>ROUGE-L F1</b> against the reference summaries (<code>highlights</code> field) from CNN/DailyMail, computed via the <code>rouge l1 score</code> from rouge_score library.</p>





In this section, you can add **text**, **tables**, and **figures**.

## **Experimental Results**
Describe here the main experimental results. Critically discuss them. Compare them with results available in the literature (if applicable).

In the experiments, I choosed following dataset CNN/DailyMail and gov_report,qmsum, vcsum from LongBench.



<div style="display: flex; gap: 5px; justify-content: center; margin: 1rem 0;">
  <figure style="text-align: center; flex: 1;">
    <img src="../plots/h2o_coverage_vs_budget.png" width="900">
    <figcaption>Figure 4: budget sweep and Rouge-L score</figcaption>
  </figure>
  <figure style="text-align: center; flex: 1;">
    <img src="../plots/H2O paper CNN plots.png" width="750">
    <figcaption>Figure 5: Plots in the H2O paper</figcaption>
  </figure>
</div>

The original H2O paper reports roughly 80% of full-cache accuracy at a 20% compression ratio on the CNN/DailyMail dataset, and our reproduction achieves comparable results. Figure 5 presents two baselines: full cache (no compression) and recent-only cache. Figure 4 compares five configurations: (1) random KV cache eviction, (2) local/recent-only cache, (3) full cache with no eviction, (4) H2O with per-layer eviction (solid line), and (5) H2O with per-head eviction (dashed line).

At 80% budget (512 × 80%), the local method performs worst and random eviction second worst. As the budget decreases, H2O configurations that allocate a smaller fraction to the recent window and a larger fraction to heavy hitters achieve higher ROUGE-L scores at the same total budget, illustrating the importance of the heavy-hitter selection mechanism. Across all budget levels, per-head eviction consistently outperforms per-layer eviction by 4–6 ROUGE-L points. Although all compressed methods fall below the full-cache baseline, the results clearly demonstrate the trade-off between total budget size and the allocation split between recent tokens and heavy hitters.




If we take the best among all different total budget we got :
  <figure style="text-align: center; flex: 1;">
    <img src="../plots/coverage_vs_budget.png" width="750">
    <figcaption>Figure 6: Roug-L vesus Budget for all methods</figcaption>
  </figure>


We can see that, we can further use the ratio between recent and heavy hitter portion ratio to fine tune and get the best performance of all different total budget, but after 20% compression ratio the accuracy drops sharply.


  <figure style="text-align: center; flex: 1;">
    <img src="../plots/rouge_all_configs_bar.png" width="750">
    <figcaption>Figure 6: All Roug-L for all methods</figcaption>
  </figure>

## Latency test

I did a latency test, but the timer starts before the eviction. So this latency is prefill + eviction latency.

  <figure style="text-align: center; flex: 1;">
    <img src="../plots/latency_memory_vs_budget.png" width="1200">
    <figcaption>Figure 7: Latency comparision</figcaption>
  </figure>


## Qualitative Results: KV Cache Eviction on CNN/DailyMail

Then I am very curious about the output sentence, so I take a test sample which is the index 0 for validation set Validation[0] and checked the outputs:

**Here is the test sample:**

**Article**

(CNN) Share, and your gift will be multiplied. That may sound like an esoteric adage, but when Zully Broussard selflessly decided to give one of her kidneys to a stranger, her generosity paired up with big data — resulting in six patients receiving transplants.
"I thought I was going to help this one person who I don't know, but the fact that so many people can have a life extension, that's pretty big," Broussard told CNN affiliate KGO.
The power that multiplied Broussard's gift was data processing of genetic profiles from donor-recipient pairs. Working on a swapping principle scaled to a much higher level, California Pacific Medical Center in San Francisco coordinated five surgeons, dozens of support staff, and more than 40 personnel to perform surgeries on 12 people — extracting six kidneys from donors and implanting them into six recipients.
A computer programmer named David Jacobs, himself a kidney transplant recipient, created MatchGrid, a program that genetically matches donor pairs or chains in weeks rather than months. With altruistic donor Broussard entering the chain, the number of viable matching options jumped from 3–4 to 140.

**Reference Summary**

Zully Broussard decided to give a kidney to a stranger.A new computer program helped her donation spur transplants for six kidney patients.

---

| Config | Strategy | ROUGE-L |
|--------|----------|---------|
| `full` | per_head | 0.1714 |
| `h2o_b7_r5` | layer_shared | 0.1818 |
| `h2o_b6_r3` | layer_shared | 0.1667 |
| `h2o_b5_r1` | layer_shared | 0.1584 |
| `random_b1` | per_head | 0.0488 |
| `local_b4` | layer_shared | 0.0325 |
| `local_b1` | layer_shared | 0.0165 |

---

### Full Cache — `full` · per_head · ROUGE-L = 0.1714

```
A woman named Zully Broussard selflessly donated one of her kidneys to a stranger,
which sparked a chain reaction of six transplants, with her kidney being matched with
a recipient who was paired with a donor. This rare and complex process, known as paired
donation, uses genetic profiles to match compatible donors and recipients, and has been
made possible by the use of computer programs like MatchGrid...
```

---

### H2O — `h2o_b7_r5` · layer_shared · ROUGE-L = 0.1818

```
A selfless act of generosity led to a chain of six kidney transplants, with the donor's
kidney being matched with a stranger's recipient. The chain was made possible by a
computer program called MatchGrid, which uses genetic data to find compatible donors
and recipients. The program was developed by a programmer who used his skills to create
a system that can quickly match up donors and recipients...
```

---

### H2O — `h2o_b6_r3` · layer_shared · ROUGE-L = 0.1667

```
A woman named Zully Broussard selflessly donated one of her kidneys to a stranger,
which was then matched with another donor to save the life of a patient. The process,
known as paired donation, uses genetic profiles to find compatible donors and recipients,
and has been successful in pairing donors and recipients in a remarkable chain of surgeries...
```

---

### H2O — `h2o_b5_r1` · layer_shared · ROUGE-L = 0.1584

```
A selfless act of generosity led to a remarkable chain of events, where a kidney was
successfully transplanted from one person to another through a process called paired
donation. The process involves finding a compatible donor and recipient, and then using
data processing to match them...
```

---

### Random Eviction — `random_b1` · per_head · ROUGE-L = 0.0488

```
A A A A A A A A A A A A A AA AA AA AA AA AA AA AA AA AA AA AA AA AA AA
AA AA AA AA AA AA AA AA AA AA AA AA AA AA AA AA AA AA AA AA AA AA AA ...
```

Severe repetition — random eviction at 10% budget destroys output coherence entirely.

---

### Local/Sliding Window — `local_b4` · layer_shared · ROUGE-L = 0.0325

```
A nice Shirley Williams her my my my my my my my my my my my my my my
my my my my my my my my my my my my my my my my my my my my my my ...
```

Degenerate output — sliding window discards long-range context, causing the model
to latch onto the last few tokens in the article.

---

### Local/Sliding Window — `local_b1` · layer_shared · ROUGE-L = 0.0165

```
A angel my divine divine divine divine divine divine divine divine divine
divine divine divine divine divine divine divine divine divine divine ...
```

Complete failure — smallest budget combined with recent-only eviction produces
fully incoherent output.

---

### Key Observations

- H2O at high budget (`b7_r5`) slightly exceeds full cache (0.1818 vs. 0.1714),
  suggesting heavy-hitter selection can filter noise from the full context.
- Heavy hitter allocation is critical: `h2o_b5_r1` (50% budget, small recent window)
  outperforms `local_b4` (40% budget, all recent) by 4.8x in ROUGE-L.
- Random eviction collapses at low budget, producing no coherent output.
- Local/sliding window is the most fragile method, failing even at moderate budgets
  due to loss of long-range narrative context.



  <figure style="text-align: center; flex: 1;">
    <img src="../plots/prediction_comparison_sample0.png" width="100%">
    <figcaption>Figure 7: Complete rouge-L score for sample 0 in validation set</figcaption>
  </figure>






In this section, you can add **text** and **figures**, **tables**, **plots**, and code. Make sure the code is runnable and replicable.

#*Code Section*



Import all dependencies;

Treansformers version is **5.0.0**

Torch version is **2.10.0**

**Model**:*Llama-3.2-1B-Instruct*

**Dataset**:*cnn_dailymail-3.0.0 or longbench*



Model set up and arguments parse
Command format is :

The script requires four key inputs to run:


- **--dataset**: Which dataset to use.(eg."cnn_dailymail", "gov_report", "qmsum", "vcsum")

- **--num-samples**: How many data points to test.

- **--split**: Which part of the dataset to use (e.g., **validation**).

- **--kv-mode**: The specific compression configuration (e.g., **h2o_b3_r1** this means 30% of the cache size, 10% of cache size used as recent portion).

- **--h2o-strategy**: Whether to calculate importance per individual head (**per_head**) or shared across the layer (**layer_shared**).

- **--max-seq-len**: max input sequence length.

- **--cache-size**: max cache budget( <b>100% budget</b> ).

## **Conclusions**

Summarize what you could and could not conclude based on your experiments.
In this section, you can add **text**.



## **References**
You can add here the citations of books, websites, or academic papers, etc.

### References

1.  Zhang, Z., Sheng, Y., Zhou, T., Chen, T., Zheng, L., Cai, R., ... & Chen, B.
(2023). **H2O: Heavy-Hitter Oracle for Efficient Generative Inference of Large Language Models**. *arXiv preprint arXiv:2306.14048*. [https://arxiv.org/abs/2306.14048](https://arxiv.org/abs/2306.14048)

2.  Yilong Chen, Guoxia Wang, Junyuan Shang ... (2023). **NACL: A General and Effective KV Cache Eviction Framework for LLMs at Inference Time**. *arXiv preprint arXiv:2408.03675*. [https://arxiv.org/abs/2408.03675](https://arxiv.org/abs/2408.03675)

3. Li, H., Li, Y., Tian, A., Tang, T., Xu, Z., Chen, X., ... & Chen, L. (2025). **A Survey on Large Language Model Acceleration based on KV Cache Management**. *arXiv preprint arXiv:2412.19442*. [https://arxiv.org/abs/2412.19442](https://arxiv.org/abs/2412.19442)


4. Meta AI. (2024). **Llama 3.2-1B**. Hugging Face. [https://huggingface.co/meta-llama/Llama-3.2-1B](https://huggingface.co/meta-llama/Llama-3.2-1B)

5. A., Liu, P. J., & Manning, C. D. (2017). **CNN/Daily Mail Dataset**. Hugging Face. [https://huggingface.co/datasets/abisee/cnn_dailymail](https://huggingface.co/datasets/abisee/cnn_dailymail)
   
6. Bai, Yushi et all. (2023). **LongBench Dataset**. Hugging Face. [https://huggingface.co/datasets/yanbingzheng/LongBench](https://huggingface.co/datasets/yanbingzheng/LongBench)